# Proyecto Final 🐼

## Fundamentos de Python

## Profesor: Ing. Andrés Mena A.

### Nombre del estudiante: MELANY YARITZA MORALES JIMÉNEZ
* * *

In [14]:
%pip install pandas

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 26.2 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [15]:
#Import y archivo datos

import pandas as pd
import matplotlib.pyplot as plt

# Variable global
archivo_excel = "Team productivity (01.01.2026 til 08.21.2026).xlsx"

In [16]:
# Función para leer y limpiar datos
def cargar_datos():
    # Leer el archivo Excel
    df = pd.read_excel(archivo_excel)

    # Eliminar columnas completamente vacías
    df = df.dropna(axis=1, how="all")

    # Quitar espacios en los nombres de las columnas
    df.columns = df.columns.str.strip()

    # Convertir la fecha
    df["Date"] = pd.to_datetime(
        df["Date"],
        errors="coerce"
    )

    # Convertir la hora
    df["Time"] = pd.to_datetime(
        df["Time"],
        errors="coerce"
    ).dt.time

    # Eliminar filas sin fecha o persona responsable
    df = df.dropna(
        subset=["Date", "Changed by", "Full name"]
    )

    # Ordenar por fecha
    df = df.sort_values("Date")

    # Reiniciar los índices
    df = df.reset_index(drop=True)

    print("Datos cargados y limpiados correctamente")
    print(f"Cantidad de registros: {len(df)}")

    display(df.head())

    return df


# Función para contar órdenes por persona y día
def promedio_por_persona(df):
    ordenes = df[df["Transaction"] == "VA02"].copy()

    ordenes_diarias = (
        ordenes.groupby(["Date", "Changed by", "Full name"])
        .size()
        .reset_index(name="Ordenes del dia")
        .sort_values(["Date", "Full name"])
    )

    ordenes_diarias["Total acumulado"] = (
        ordenes_diarias.groupby("Changed by")["Ordenes del dia"]
        .cumsum()
    )

    ordenes_diarias["Ordenes dia anterior"] = (
        ordenes_diarias.groupby("Changed by")["Ordenes del dia"]
        .shift(1)
        .fillna(0)
        .astype(int)
    )

    ordenes_diarias["Diferencia"] = (
        ordenes_diarias["Ordenes del dia"]
        - ordenes_diarias["Ordenes dia anterior"]
    )

    total_por_persona = (
        ordenes.groupby(["Changed by", "Full name"])
        .size()
        .reset_index(name="Total de ordenes")
        .sort_values("Total de ordenes", ascending=False)
    )

    print("\nOrdenes por persona y dia:")
    display(ordenes_diarias)

    print("\nTotal de ordenes por persona:")
    display(total_por_persona)

    return ordenes_diarias, total_por_persona


# Función para calcular el porcentaje de participación

def porcentaje_participacion(df):
    ordenes = df[df["Transaction"] == "VA02"]

    participacion = (
        ordenes.groupby(["Changed by", "Full name"])
        .size()
        .reset_index(name="Total de ordenes")
    )

    total_ordenes = participacion["Total de ordenes"].sum()
    participacion["Porcentaje"] = (
        participacion["Total de ordenes"]
        / total_ordenes
        * 100
    )
    participacion = participacion.sort_values(
        "Porcentaje",
        ascending=False
    )

    participacion.plot(
        x="Full name",
        y="Porcentaje",
        kind="bar",
        legend=False,
        title="Participación de órdenes por agente"
    )
    plt.ylabel("Porcentaje del total (%)")
    plt.xlabel("Agente")
    plt.xticks(rotation=45, ha="right")
    plt.tight_layout()
    plt.show()

    display(participacion)
    return participacion


# Función para comparar las órdenes por mes

def comparacion_mensual(df):
    ordenes = df[df["Transaction"] == "VA02"].copy()
    ordenes["Mes"] = ordenes["Date"].dt.to_period("M").astype(str)

    mensual = (
        ordenes.groupby("Mes")
        .size()
        .reset_index(name="Total de ordenes")
    )

    mensual.plot(
        x="Mes",
        y="Total de ordenes",
        kind="bar",
        legend=False,
        title="Comparación mensual de órdenes"
    )
    plt.ylabel("Cantidad de órdenes")
    plt.xlabel("Mes")
    plt.xticks(rotation=45)
    plt.tight_layout()
    plt.show()

    display(mensual)
    return mensual


# Función para contar órdenes por mes
def promedio_por_mes(df):
    ordenes = df[df["Transaction"] == "VA02"].copy()
    ordenes["Mes"] = ordenes["Date"].dt.to_period("M")
    resultado = ordenes.groupby("Mes").size()
    print("\nCantidad de órdenes por mes:\n", resultado)
    return resultado


# Función para top 5 días
def top_5_dias(df):
    resultado = (
        df[df["Transaction"] == "VA02"]
        .groupby("Date")
        .size()
        .sort_values(ascending=False)
        .head(5)
    )
    print("\nTop 5 días con más órdenes:\n", resultado)
    return resultado

In [17]:
# Programa principal con interacción
def main():
    df = cargar_datos()

    while True:
        print("\n--- Menú de análisis ---")
        print("1. Órdenes por persona y día")
        print("2. Cantidad de órdenes por mes")
        print("3. Top 5 días con más órdenes")
        print("4. Graficar órdenes por persona")
        print("5. Porcentaje de participación por agente")
        print("6. Comparación mensual")
        print("7. Salir")

        opcion = input("Seleccione una opción: ")

        if opcion == "1":
            promedio_por_persona(df)
        elif opcion == "2":
            promedio_por_mes(df)
        elif opcion == "3":
            top_5_dias(df)
        elif opcion == "4":
            _, total_por_persona = promedio_por_persona(df)
            total_por_persona.plot(
                x="Full name",
                y="Total de ordenes",
                kind="bar",
                legend=False,
                title="Total de órdenes por persona"
            )
            plt.ylabel("Cantidad de órdenes")
            plt.xlabel("Persona")
            plt.xticks(rotation=45, ha="right")
            plt.tight_layout()
            plt.show()
        elif opcion == "5":
            porcentaje_participacion(df)
        elif opcion == "6":
            comparacion_mensual(df)
        elif opcion == "7":
            print("Saliendo del programa...")
            break
        else:
            print("Opción inválida, intente de nuevo.")


# Ejecutar
main()

Datos cargados y limpiados correctamente
Cantidad de registros: 1943


,Unnamed: 0,Selection number,Date,Time,Changed by,Full name,Transaction,Name of transaction,Table,Table description,Table field,Field Label,Old value,New value,Data record,Log number,Table key,Program name,Host name,Language Key of Text Environment
0,@0S\QFurther information@,1,2026-01-22,NaT,30802358,Annath Fernandez,VA02,Change Sales Order,TCVIEW,Table control views (user settings),TCUDEFAULT,Default variant of a table control,NaN,UD,changed,222005snpr1ap11166,ABAP Program Name: SAPMV45A; User Type: B; Use...,SAPMV45A,snpr1ap1,EN
1,@0S\QFurther information@,1,2026-01-22,NaT,30802358,Annath Fernandez,VA02,Change Sales Order,TCVIEW,Table control views (user settings),TCUDEFAULT,Default variant of a table control,NaN,UD,changed,424259snpr1ap11166,ABAP Program Name: SAPMV45A; User Type: B; Use...,SAPMV45A,snpr1ap1,EN
2,@0S\QFurther information@,1,2026-01-22,NaT,30802358,Annath Fernandez,VA02,Change Sales Order,TCVIEW,Table control views (user settings),KEY,Data record inserted,NaN,NaN,inserted,160115snpr1ap11166,ABAP Program Name: SAPMV45A; User Type: B; Use...,SAPMV45A,snpr1ap1,EN
3,@0S\QFurther information@,1,2026-01-22,NaT,30802358,Annath Fernandez,VA02,Change Sales Order,TCVIEW,Table control views (user settings),TCUDEFAULT,Default variant of a table control,NaN,UD,changed,420416snpr1ap11166,ABAP Program Name: SAPMV45A; User Type: B; Use...,SAPMV45A,snpr1ap1,EN
4,@0S\QFurther information@,1,2026-01-22,NaT,30802358,Annath Fernandez,VA02,Change Sales Order,TCVIEW,Table control views (user settings),KEY,Data record inserted,NaN,NaN,inserted,159874snpr1ap11166,ABAP Program Name: SAPMV45A; User Type: B; Use...,SAPMV45A,snpr1ap1,EN



--- Menú de análisis ---
1. Órdenes por persona y día
2. Cantidad de órdenes por mes
3. Top 5 días con más órdenes
4. Graficar órdenes por persona
5. Porcentaje de participación por agente
6. Comparación mensual
7. Salir

Cantidad de órdenes por mes:
 Mes
2026-01     350
2026-02    1575
2026-04      10
2026-05       8
Freq: M, dtype: int64

--- Menú de análisis ---
1. Órdenes por persona y día
2. Cantidad de órdenes por mes
3. Top 5 días con más órdenes
4. Graficar órdenes por persona
5. Porcentaje de participación por agente
6. Comparación mensual
7. Salir
Opción inválida, intente de nuevo.

--- Menú de análisis ---
1. Órdenes por persona y día
2. Cantidad de órdenes por mes
3. Top 5 días con más órdenes
4. Graficar órdenes por persona
5. Porcentaje de participación por agente
6. Comparación mensual
7. Salir
Saliendo del programa...
